# M19 · RAG & query understanding

Curriculum · Domain 4 · LLMs

**Ground answers in retrieved evidence, then fallback when confidence is low.**

We build a tiny retrieval pipeline and slot parser. Retrieval uses cosine similarity

$$s_i=\frac{q^\top c_i}{\|q\|\|c_i\|}$$

so all numbers are inspectable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

rng = np.random.default_rng(19)

## A tiny evidence corpus

Each chunk is a short piece of product knowledge we might cite for Creator Marketplace or ads relevance.

In [ ]:
chunks = [
    "B2B cybersecurity creators often discuss enterprise risk compliance and cloud security",
    "Canada finance executives respond to creators with regional market expertise",
    "Fitness creators focus on workouts nutrition and consumer wellness",
    "Search ads relevance depends on query intent landing page match and policy safety",
    "Creative intelligence prompts improve headlines with brand claims and evidence",
]

query = "enterprise security creators in Canada for finance audience"

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(chunks + [query]).toarray()
chunk_vecs = X[:-1]
query_vec = X[-1]

print(chunk_vecs.shape)

assert chunk_vecs.shape[0] == len(chunks)

## Cosine retrieval

We normalize vectors and score every chunk against the query. Higher cosine means the chunk points in a more similar direction.

In [ ]:
def normalize_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)

chunk_unit = normalize_rows(chunk_vecs)
query_unit = query_vec / max(np.linalg.norm(query_vec), 1e-12)
scores = chunk_unit @ query_unit
ranked = np.argsort(scores)[::-1]

for idx in ranked:
    print(round(scores[idx], 3), chunks[idx])

assert scores[ranked[0]] >= scores[ranked[1]]

## Confidence threshold

A RAG system should not answer just because it found something. We compare the best score to a threshold $\gamma$ and fallback if retrieval is weak.

In [ ]:
gamma = 0.20
best_idx = ranked[0]
best_score = scores[best_idx]
should_answer = best_score >= gamma

print("best score", round(best_score, 3))
print("answer?", should_answer)

assert should_answer

## Simple NL to structured slots

This parser is intentionally small. Production systems use learned parsers and validation, but the shape is the same: intent, slots, confidence, fallback.

In [ ]:
def parse_creator_query(text):
    lower = text.lower()
    topic = "security" if "security" in lower else None
    geo = "Canada" if "canada" in lower else None
    audience = "finance" if "finance" in lower else None
    filled = sum(value is not None for value in [topic, geo, audience])
    confidence = filled / 3.0
    return {
        "intent": "creator_search",
        "topic": topic,
        "geo": geo,
        "audience": audience,
        "confidence": confidence,
    }

parsed = parse_creator_query(query)

print(parsed)

assert parsed["confidence"] == 1.0

## Combine retrieval and parsing

A robust answer needs both enough evidence and enough parsing confidence.

In [ ]:
parse_threshold = 0.75
retrieval_ok = best_score >= gamma
parse_ok = parsed["confidence"] >= parse_threshold
action = "answer_with_citation" if retrieval_ok and parse_ok else "fallback"

print(action)
print("citation:", chunks[best_idx])

assert action == "answer_with_citation"

## Evaluate recall@k

Suppose chunks 0 and 1 are the relevant evidence for this query. Recall@k asks how many of those relevant chunks appear in the top $k$.

In [ ]:
relevant = {0, 1}
top_k = set(ranked[:3])
recall_at_3 = len(relevant.intersection(top_k)) / len(relevant)

print("recall@3", recall_at_3)

assert recall_at_3 >= 0.5

## Visualize scores

A bar chart makes the retrieval margin easy to see. Small margins are a good reason to rerank or fallback.

In [ ]:
labels = [f"chunk {idx}" for idx in range(len(chunks))]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(labels, scores, color="#4c78a8")
ax.axhline(gamma, color="#f58518", linestyle="--", label="threshold")
ax.set_ylabel("cosine score")
ax.set_title("retrieval scores")
ax.legend()
plt.show()

## Practice

1. Raise `gamma` until the system falls back.
2. Add a new chunk about Canadian cybersecurity creators and rerun retrieval.
3. Modify `parse_creator_query` to extract `industry` separately from `audience`.

In [ ]:
# Your turn:
